# 05 - Evaluation Diagnostic

Diagnostic test for evaluation pipeline (20 examples only).

**Purpose:** Verify model loading, prediction generation, and metric computation before full evaluation.

**Runtime:** ~5 minutes

In [ ]:
%%capture
!pip install unsloth rouge_score bert_score

In [ ]:
import os
import sys
import json
import time

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

# Setup logging
log_path = "results/eval_diagnostic_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Evaluation diagnostic started")

In [ ]:
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Load config
with open("configs/base.yaml") as f:
    config = yaml.safe_load(f)

EXPERIMENT_NAME = "v1_baseline"
MODEL_NAME = config["model"]["name"]
MAX_SEQ_LENGTH = config["model"]["max_seq_length"]
DIAGNOSTIC_SIZE = 20  # Only 20 examples for diagnostic

log(f"Config loaded:")
log(f"  Model: {MODEL_NAME}")
log(f"  Max seq length: {MAX_SEQ_LENGTH}")
log(f"  Experiment: {EXPERIMENT_NAME}")
log(f"  Diagnostic size: {DIAGNOSTIC_SIZE}")

# Run data pipeline if needed
if not os.path.exists("data/splits/test.json"):
    log("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

# Load test data (only first 20 examples)
with open("data/splits/test.json") as f:
    test_data = json.load(f)[:DIAGNOSTIC_SIZE]

log(f"\nTest set: {len(test_data)} examples (diagnostic)")
task_counts = Counter(ex.get("task_type", "unknown") for ex in test_data)
for task, count in sorted(task_counts.items()):
    log(f"  {task}: {count}")

In [ ]:
from unsloth import FastLanguageModel

def generate_response(model, tokenizer, messages, max_new_tokens=512):
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)

def build_messages(conversations):
    messages = []
    for turn in conversations:
        role = turn["from"]
        if role == "gpt":
            role = "assistant"
        messages.append({"role": role, "content": turn["value"]})
    return messages[:-1]

log("Helper functions defined.")

In [ ]:
log("\n=== Loading Base Model ===")
t0 = time.time()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

t1 = time.time()
log(f"Base model loaded in {t1-t0:.1f}s")
log(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Generate base predictions
log("\n=== Generating Base Predictions ===")
base_predictions = []
t0 = time.time()

for i, example in enumerate(test_data):
    messages = build_messages(example["conversations"])
    pred = generate_response(base_model, base_tokenizer, messages)
    base_predictions.append(pred)
    if (i + 1) % 5 == 0:
        log(f"  Base model: {i+1}/{len(test_data)}")

t1 = time.time()
log(f"Base predictions done: {len(base_predictions)} in {t1-t0:.1f}s")

# Free GPU memory
del base_model
del base_tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()
log("GPU memory freed.")

In [ ]:
log("\n=== Downloading Fine-tuned Model ===")
t0 = time.time()

# Download training output from Kaggle
import subprocess
result = subprocess.run(
    ["kaggle", "kernels", "output", "ericwang7717/fingpt-training", "-p", "outputs/training_output"],
    capture_output=True, text=True
)
if result.returncode == 0:
    log("Training output downloaded successfully")
else:
    log(f"Download failed: {result.stderr}")

t1 = time.time()
log(f"Download time: {t1-t0:.1f}s")

# Load fine-tuned model (base + LoRA adapter)
log("\n=== Loading Fine-tuned Model (Base + LoRA) ===")
lora_adapter_path = "outputs/training_output/outputs/v1_baseline/lora_adapter"
log(f"Loading base model: {MODEL_NAME}")
log(f"Loading LoRA adapter from: {lora_adapter_path}")
t0 = time.time()

# First load base model
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Then load LoRA adapter
from peft import PeftModel
ft_model = PeftModel.from_pretrained(ft_model, lora_adapter_path)
ft_model = ft_model.merge_and_unload()  # Merge LoRA into base for faster inference
FastLanguageModel.for_inference(ft_model)

t1 = time.time()
log(f"Fine-tuned model loaded in {t1-t0:.1f}s")
log(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import re

def extract_sentiment(text):
    match = re.search(r"\*\*Sentiment:\s*(\w+)\*\*", text, re.IGNORECASE)
    if match:
        label = match.group(1).capitalize()
        if label in ("Positive", "Negative", "Neutral"):
            return label
    text_lower = text.lower()
    if "positive" in text_lower and "negative" not in text_lower:
        return "Positive"
    if "negative" in text_lower and "positive" not in text_lower:
        return "Negative"
    return "Neutral"

def compute_sentiment_metrics(predictions, references):
    pred_labels = [extract_sentiment(p) for p in predictions]
    ref_labels = [extract_sentiment(r) for r in references]
    correct = sum(1 for p, r in zip(pred_labels, ref_labels) if p == r)
    return {"accuracy": correct / len(ref_labels), "correct": correct, "total": len(ref_labels)}

def compute_rouge_l(predictions, references):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(predictions, references)]
    return {"rouge_l_f1": np.mean(scores), "std": np.std(scores)}

log("Metric functions defined.")

In [ ]:
log("\n=== Computing Metrics ===")

metrics_table = []

for task_type in sorted(set(r["task_type"] for r in results)):
    task_results = [r for r in results if r["task_type"] == task_type]
    if not task_results:
        continue
    
    refs = [r["reference"] for r in task_results]
    base_preds = [r["base_prediction"] for r in task_results]
    ft_preds = [r["ft_prediction"] for r in task_results]
    
    if task_type == "sentiment":
        base_m = compute_sentiment_metrics(base_preds, refs)
        ft_m = compute_sentiment_metrics(ft_preds, refs)
        metrics_table.append({
            "Task": task_type,
            "Metric": "Accuracy",
            "Base": f"{base_m['accuracy']:.3f}",
            "Fine-tuned": f"{ft_m['accuracy']:.3f}",
            "Delta": f"{ft_m['accuracy'] - base_m['accuracy']:+.3f}",
        })
    
    # ROUGE-L for all tasks
    try:
        base_rouge = compute_rouge_l(base_preds, refs)
        ft_rouge = compute_rouge_l(ft_preds, refs)
        metrics_table.append({
            "Task": task_type,
            "Metric": "ROUGE-L",
            "Base": f"{base_rouge['rouge_l_f1']:.3f}",
            "Fine-tuned": f"{ft_rouge['rouge_l_f1']:.3f}",
            "Delta": f"{ft_rouge['rouge_l_f1'] - base_rouge['rouge_l_f1']:+.3f}",
        })
    except Exception as e:
        log(f"ROUGE-L failed for {task_type}: {e}")

df_metrics = pd.DataFrame(metrics_table)
log("\n" + "="*70)
log("DIAGNOSTIC EVALUATION RESULTS")
log("="*70)
log(df_metrics.to_string(index=False))

# Save
df_metrics.to_markdown("results/comparison_table_diagnostic.md", index=False)
log(f"\nSaved to results/comparison_table_diagnostic.md")

In [ ]:
log("\n=== Example Outputs ===")

for task_type in sorted(set(r["task_type"] for r in results)):
    task_results = [r for r in results if r["task_type"] == task_type][:3]
    
    log(f"\n{'='*70}")
    log(f"Task: {task_type}")
    log(f"{'='*70}")
    
    for i, r in enumerate(task_results):
        log(f"\n--- Example {i+1} ---")
        log(f"Reference: {r['reference'][:200]}...")
        log(f"\nBase Model:\n{r['base_prediction'][:300]}...")
        log(f"\nFine-tuned:\n{r['ft_prediction'][:300]}...")

In [ ]:
log("\n" + "="*70)
log("DIAGNOSTIC SUMMARY")
log("="*70)

log(f"\nTest examples: {len(test_data)}")
log(f"Results saved: results/eval_outputs/{EXPERIMENT_NAME}_diagnostic.json")
log(f"Metrics saved: results/comparison_table_diagnostic.md")
log(f"Log saved: {log_path}")

log("\n✅ Diagnostic complete. If results look good, run full evaluation.")
log("   Use 05_evaluation.ipynb for full evaluation (1508 examples).")